# Cleaning

**Imports**

In [46]:
# Imports
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

**Loading Raw Data**

In [47]:
# Loading raw data
reviews_df = pd.read_csv("../data/reviews.csv.gz", low_memory=False)
listings_df = pd.read_csv("../data/listings.csv.gz", low_memory=False)


**Checking for Duplicates**

In [48]:
# Check for duplicates
print(f"Reviews DataFrame contains duplicates: {reviews_df.duplicated().any()}")
print(f"Listings DataFrame contains duplicates: {listings_df.duplicated().any()}")
print(f"Listings DataFrame contains listing id duplicates: {listings_df['id'].duplicated().any()}")

Reviews DataFrame contains duplicates: False
Listings DataFrame contains duplicates: False
Listings DataFrame contains listing id duplicates: False


**Drop Duplicates and Useless Columns**

In [49]:
# Drop duplicate
reviews_df = reviews_df.drop_duplicates(subset=['listing_id', 'reviewer_id', 'date', 'comments'], keep='first')

reviews_df = reviews_df.drop(columns=['id', 'reviewer_name'])
print(reviews_df.columns)

# Parse date
reviews_df['date'] = pd.to_datetime(reviews_df['date'], errors='coerce')

# Check for missing comments values
print(f"Missing comments values: {reviews_df['comments'].isnull().sum()}")

# Clean comments
reviews_df['comments'] = reviews_df['comments'].fillna("")
reviews_df['comments'] = reviews_df['comments'].astype(str)
reviews_df['comments'] = reviews_df['comments'].str.replace(r'[\n\r]+', ' ', regex=True)
reviews_df['comments'] = reviews_df['comments'].str.strip()

# Drop empty comments
reviews_df = reviews_df[reviews_df['comments'] != ""]

Index(['listing_id', 'date', 'reviewer_id', 'comments'], dtype='str')
Missing comments values: 110


**listings.csv and reviews.csv**

In [50]:
# Unique IDs in each file
review_ids = set(reviews_df['listing_id'].unique())
listing_ids = set(listings_df['id'].unique())

# Overlap analysis
combined_ids = review_ids & listing_ids 
only_reviews = review_ids - listing_ids
only_listings = listing_ids - review_ids

# length before filtering
total_reviews_before = len(reviews_df)

# Filter reviews
reviews_df = reviews_df[reviews_df['listing_id'].isin(listing_ids)] 


print(f"Total unique properties in reviews.csv:   {len(review_ids)}")
print(f"Total unique properties in listings.csv:  {len(listing_ids)}")
print(f"Properties with at least 1 comment:       {len(combined_ids)}")
print(f"Properties with zero comments:            {len(only_listings)}")
print(f"Properties with reviews but no listing:   {len(only_reviews)}")

print(f"\nTotal review comments after and before filtering:   {total_reviews_before} / {len(reviews_df)}")

Total unique properties in reviews.csv:   14421
Total unique properties in listings.csv:  19410
Properties with at least 1 comment:       14421
Properties with zero comments:            4989
Properties with reviews but no listing:   0

Total review comments after and before filtering:   1019151 / 1019151


**Unique Reviewers**

In [51]:
# Total unique reviewers
total_unique_reviewers = reviews_df['reviewer_id'].nunique()
print(f"Total unique reviewers: {total_unique_reviewers} / {len(reviews_df)}")

# how many reviews each guest has left
reviewer_counts = reviews_df.groupby('reviewer_id').size().reset_index(name='review_count')

# Group for distribution 
distribution = reviewer_counts['review_count'].value_counts().sort_index().head(25)

print("\nReviewer Distribution")
for count, num_people in distribution.items():
    print(f"{num_people} guests left exactly {count} review(s)")

print(f"{len(reviewer_counts[reviewer_counts['review_count'] > 20])} guests left more than 20 reviews")

print(f"\n Top 10")
print(reviewer_counts.sort_values('review_count', ascending=False).head(25).to_string(index=False))

Total unique reviewers: 963919 / 1019151

Reviewer Distribution
919715 guests left exactly 1 review(s)
37318 guests left exactly 2 review(s)
4784 guests left exactly 3 review(s)
1224 guests left exactly 4 review(s)
452 guests left exactly 5 review(s)
196 guests left exactly 6 review(s)
85 guests left exactly 7 review(s)
46 guests left exactly 8 review(s)
34 guests left exactly 9 review(s)
26 guests left exactly 10 review(s)
8 guests left exactly 11 review(s)
8 guests left exactly 12 review(s)
7 guests left exactly 13 review(s)
1 guests left exactly 14 review(s)
2 guests left exactly 15 review(s)
3 guests left exactly 16 review(s)
1 guests left exactly 18 review(s)
2 guests left exactly 19 review(s)
1 guests left exactly 20 review(s)
1 guests left exactly 21 review(s)
1 guests left exactly 22 review(s)
1 guests left exactly 23 review(s)
1 guests left exactly 24 review(s)
1 guests left exactly 26 review(s)
1 guests left exactly 28 review(s)
6 guests left more than 20 reviews

 Top 10
 re

**Reviewers Weight**

In [ ]:
# Count unique properties each reviewer has reviewed
reviewer_experience = reviews_df.groupby("reviewer_id")["listing_id"].nunique().reset_index(
    name="unique_properties"
)

# Log-scaled weight
reviewer_experience["reviewer_weight"] = np.log1p(reviewer_experience["unique_properties"])

reviewer_experience.to_csv("../data/generated/reviewer_experience_weighted.csv", index=False)

print(f"Total reviewers: {len(reviewer_experience)}")
print(f"\nWeight distribution:")
print(reviewer_experience["reviewer_weight"].describe())
print(f"\nSample:")
print(reviewer_experience.sort_values("unique_properties", ascending=False).head(1000))

Total reviewers: 963919

Weight distribution:
count    963919.000000
mean          0.713035
std           0.099035
min           0.693147
25%           0.693147
50%           0.693147
75%           0.693147
max           3.178054
Name: reviewer_weight, dtype: float64

Sample:
        reviewer_id  unique_properties  reviewer_weight
945612    655892833                 23         3.178054
20771       2570702                 21         3.091042
252986     41612345                 21         3.091042
136424     19267544                 19         2.995732
254361     41925397                 19         2.995732
...             ...                ...              ...
222838     34767732                  4         1.609438
738453    318298783                  4         1.609438
754589    342740193                  4         1.609438
343476     65146816                  4         1.609438
128139     17703361                  4         1.609438

[1000 rows x 3 columns]


# Comments - Pretrained Model

**Language Distribution**

In [53]:
from langdetect import detect
from collections import Counter

def safe_detect(text):
    try:
        return detect(text) if len(str(text)) > 10 else "too_short"
    except:
        return "unknown"

sample = reviews_df["comments"].sample(5000, random_state=42)
langs = sample.apply(safe_detect)

print("Language distribution (sample of 5000):")
for lang, count in langs.value_counts().head(15).items():
    print(f"  {lang}: {count} ({count/50:.1f}%)")

# comments that are unknown or too short
short_count = (langs == "too_short").sum()
unknown_count = (langs == "unknown").sum()
print(f"\nComments too short: {short_count} ({short_count/50:.1f}%)")
print(f"Comments unknown: {unknown_count} ({unknown_count/50:.1f}%)")

Language distribution (sample of 5000):
  en: 2966 (59.3%)
  es: 625 (12.5%)
  fr: 525 (10.5%)
  de: 199 (4.0%)
  it: 153 (3.1%)
  too_short: 122 (2.4%)
  pt: 84 (1.7%)
  ko: 60 (1.2%)
  nl: 60 (1.2%)
  zh-cn: 37 (0.7%)
  ru: 18 (0.4%)
  da: 17 (0.3%)
  pl: 17 (0.3%)
  tr: 11 (0.2%)
  no: 11 (0.2%)

Comments too short: 122 (2.4%)
Comments unknown: 1 (0.0%)


**Loading Pretrained Model - BERT**

In [54]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

# Star weights for weighted average (stays on same device as model)
stars = torch.arange(1, 6, dtype=torch.float32).to(device)

print("Model loaded!")

Using device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded!


**Scoring function**

In [55]:
def score_sentiment(texts, batch_size=64):
    """Score a list of texts and return sentiment scores (1.0 - 5.0)."""
    scores = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Scoring"):
        batch = texts[i:i + batch_size]
        batch = [str(t) for t in batch]
        
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True
        )
        # Move inputs to same device as model
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
        
        probs = torch.softmax(outputs.logits, dim=1)
        # Weighted average: [0.1, 0.05, 0.1, 0.25, 0.5] × [1,2,3,4,5] = 4.05
        batch_scores = (probs * stars).sum(dim=1).cpu().tolist()
        scores.extend(batch_scores)
    
    return scores

**Run on all Reviews**

In [56]:
# Convert to list
comments_list = reviews_df["comments"].tolist()

print(f"Scoring {len(comments_list)} reviews...")
reviews_df["sentiment_score"] = score_sentiment(comments_list, batch_size=256)

print(f"\nDone! Score distribution:")
print(reviews_df["sentiment_score"].describe())

Scoring 1019151 reviews...


Scoring: 100%|██████████| 3982/3982 [2:11:02<00:00,  1.97s/it]  



Done! Score distribution:
count    1.019151e+06
mean     4.373731e+00
std      6.137866e-01
min      1.016235e+00
25%      4.220937e+00
50%      4.578692e+00
75%      4.764724e+00
max      4.990940e+00
Name: sentiment_score, dtype: float64


**Check**

In [57]:
# Check
print("Highest Scores (positive)")
top = reviews_df.nlargest(5, "sentiment_score")[["comments", "sentiment_score"]]
for _, row in top.iterrows():
    print(f"  [{row['sentiment_score']:.2f}] {row['comments'][:100]}")

print("\nLowest Scores (negative)")
bottom = reviews_df.nsmallest(5, "sentiment_score")[["comments", "sentiment_score"]]
for _, row in bottom.iterrows():
    print(f"  [{row['sentiment_score']:.2f}] {row['comments'][:100]}")

Highest Scores (positive)
  [4.99] Five stars all around. ( one of the best we’ve stayed at ) Wonderful new flat , well equipped, comfo
  [4.99] Fantastic place to stay in overall!<br/>Highly recommended!!<br/>FIVE STARS!
  [4.99] I would highly recommend staying at Mirella's place. It is in a wonderful location, clean, and affor
  [4.99] Five Stars across the board.  The apartment was perfect and even better than expected.  Well appoint
  [4.99] The BEST!  Five stars to this fabulous accommodation in one of the best locations in Barcelona.  Bea

Lowest Scores (negative)
  [1.02] ⭐ 1 Star – Avoid This Apartment – Unsafe, Poorly Managed, and Hostile<br/>This was the worst Airbnb 
  [1.02] Schreckliches Erlebnis!!!!! 0 Sterne. Die Wohnung war die reinste Katastrophe. Bin geschockt dass ke
  [1.02] one star hotel
  [1.02] Absolutely TERRIBLE host , I wish I could give 0 stars <br/>Landed in Spain after 2 hours she was ve
  [1.02] Absolutely awful experience. Could not check in and you did

# Feature Aggregation

**Load scored Reviews**

In [63]:
scored_reviews = pd.read_csv("../data/reviews_scored.csv")
scored_reviews['date'] = pd.to_datetime(scored_reviews['date'], errors='coerce')

# Weighted product for reviewer influence
scored_reviews['weighted_product'] = scored_reviews['sentiment_score'] * scored_reviews['reviewer_weight']

# Recent vs older split (12 months)
max_date = scored_reviews['date'].max()
cutoff_date = max_date - pd.DateOffset(months=12)
recent = scored_reviews[scored_reviews['date'] >= cutoff_date]
older = scored_reviews[scored_reviews['date'] < cutoff_date]

print(f"Total: {len(scored_reviews)}, Recent: {len(recent)}, Older: {len(older)}")
print(f"Cutoff date: {cutoff_date.date()}")

Total: 1260889, Recent: 267347, Older: 993542
Cutoff date: 2024-09-15


**Aggregation**

In [ ]:
agg_data = scored_reviews.groupby('listing_id').agg(
    mean_sentiment=('sentiment_score', 'mean'),
    median_sentiment=('sentiment_score', 'median'),
    sum_weighted_product=('weighted_product', 'sum'),
    sum_weights=('reviewer_weight', 'sum'),
    min_sentiment=('sentiment_score', 'min'),
    max_sentiment=('sentiment_score', 'max'),
    std_sentiment=('sentiment_score', 'std'),
    total_reviews_scored=('sentiment_score', 'count'),
    first_review_date=('date', 'min'),
    last_review_date=('date', 'max'),
).reset_index()

# Weighted mean
agg_data['weighted_mean_sentiment'] = agg_data['sum_weighted_product'] / agg_data['sum_weights']

print(f"Aggregated {len(agg_data)} listings")
print(agg_data.head())

Aggregated 14421 listings
   listing_id  mean_sentiment  median_sentiment  sum_weighted_product  \
0       18674        4.172320          4.423029            155.895112   
1       23197        4.386101          4.520670            292.731202   
2       32711        4.337082          4.530905            471.031806   
3       34241        3.986496          4.354280             70.989815   
4       34981        4.301070          4.568413            830.085424   

   sum_weights  min_sentiment  max_sentiment  std_sentiment  \
0    37.260049       1.421857       4.881107       0.816730   
1    67.183161       1.342215       4.948233       0.587884   
2   108.602092       1.406252       4.927854       0.583005   
3    18.139610       1.026100       4.890859       1.031242   
4   193.113932       1.267600       4.975350       0.734887   

   total_reviews_scored first_review_date last_review_date  \
0                    51        2013-05-27       2025-07-31   
1                    91        2

**Range vs Negative Reviews**

In [ ]:
# Sentiment range
agg_data['sentiment_range'] = agg_data['max_sentiment'] - agg_data['min_sentiment']

# negative reviews percentage (score < 3)
pct_neg = scored_reviews.groupby('listing_id').apply(
    lambda x: (x['sentiment_score'] < 3).mean()
).reset_index(name='pct_negative')
agg_data = agg_data.merge(pct_neg, on='listing_id', how='left')

print("Sentiment range & pct_negative added")
print(agg_data[['listing_id', 'sentiment_range', 'pct_negative']].head())

Sentiment range & pct_negative added
   listing_id  sentiment_range  pct_negative
0       18674         3.459250      0.117647
1       23197         3.606018      0.032967
2       32711         3.521602      0.032895
3       34241         3.864759      0.160000
4       34981         3.707750      0.084871


**Date Based Features**

In [ ]:
# Days since last review
agg_data['days_since_last_review'] = (max_date - agg_data['last_review_date']).dt.days

# Review velocity (reviews per month)
listing_age_months = (
    (agg_data['last_review_date'] - agg_data['first_review_date']).dt.days / 30.44
).clip(lower=1)
agg_data['review_velocity'] = agg_data['total_reviews_scored'] / listing_age_months

# Recent mean sentiment (last 12 months)
recent_mean = recent.groupby('listing_id')['sentiment_score'].mean().reset_index(name='recent_mean_sentiment')
agg_data = agg_data.merge(recent_mean, on='listing_id', how='left')

# Sentiment trend (recent - older)
older_mean = older.groupby('listing_id')['sentiment_score'].mean().reset_index(name='older_mean_sentiment')
agg_data = agg_data.merge(older_mean, on='listing_id', how='left')
agg_data['sentiment_trend'] = agg_data['recent_mean_sentiment'] - agg_data['older_mean_sentiment']

print("Date features added")
print(agg_data[['listing_id', 'days_since_last_review', 'review_velocity', 'recent_mean_sentiment', 'sentiment_trend']].head())

Date features added
   listing_id  days_since_last_review  review_velocity  recent_mean_sentiment  \
0       18674                      46         0.349020               4.047131   
1       23197                       7         0.523538               4.461738   
2       32711                      38         0.900872               4.405591   
3       34241                     314         0.145451               3.639456   
4       34981                      27         1.518079               4.268593   

   sentiment_trend  
0        -0.145106  
1         0.087126  
2         0.080723  
3        -0.433800  
4        -0.035489  


**Cleanup & Split**

In [ ]:
# Fill NaNs
agg_data['std_sentiment'] = agg_data['std_sentiment'].fillna(0)
agg_data['recent_mean_sentiment'] = agg_data['recent_mean_sentiment'].fillna(agg_data['mean_sentiment'])
agg_data['sentiment_trend'] = agg_data['sentiment_trend'].fillna(0)

# Drop temp columns
agg_data = agg_data.drop(columns=[
    'sum_weighted_product', 'sum_weights',
    'first_review_date', 'last_review_date', 'older_mean_sentiment'
])

# Split into two sets
shared_cols = [
    'listing_id', 'median_sentiment', 'min_sentiment', 'max_sentiment',
    'std_sentiment', 'sentiment_range', 'pct_negative',
    'total_reviews_scored', 'days_since_last_review', 'review_velocity',
    'recent_mean_sentiment', 'sentiment_trend'
]

df_sentiment_unweighted = agg_data[shared_cols + ['mean_sentiment']].copy()

df_sentiment_weighted = agg_data[shared_cols + ['weighted_mean_sentiment']].copy()
df_sentiment_weighted = df_sentiment_weighted.rename(columns={'weighted_mean_sentiment': 'mean_sentiment'})

# Check correlation
corr, _ = pearsonr(agg_data['mean_sentiment'], agg_data['weighted_mean_sentiment'])
print(f"Correlation raw vs weighted: {corr:.4f}")
print(f"\nUnweighted shape: {df_sentiment_unweighted.shape}")
print(f"Weighted shape: {df_sentiment_weighted.shape}")
print(f"\nColumns: {list(df_sentiment_unweighted.columns)}")
print("\nSample (Weighted):")
print(df_sentiment_weighted.head())

Correlation raw vs weighted: 0.9954

Unweighted shape: (14421, 13)
Weighted shape: (14421, 13)

Columns: ['listing_id', 'median_sentiment', 'min_sentiment', 'max_sentiment', 'std_sentiment', 'sentiment_range', 'pct_negative', 'total_reviews_scored', 'days_since_last_review', 'review_velocity', 'recent_mean_sentiment', 'sentiment_trend', 'mean_sentiment']

Sample (Weighted):
   listing_id  median_sentiment  min_sentiment  max_sentiment  std_sentiment  \
0       18674          4.423029       1.421857       4.881107       0.816730   
1       23197          4.520670       1.342215       4.948233       0.587884   
2       32711          4.530905       1.406252       4.927854       0.583005   
3       34241          4.354280       1.026100       4.890859       1.031242   
4       34981          4.568413       1.267600       4.975350       0.734887   

   sentiment_range  pct_negative  total_reviews_scored  \
0         3.459250      0.117647                    51   
1         3.606018      0.

**Save CSVs**

In [ ]:
df_sentiment_unweighted.to_csv("../data/generated/sentiment_features_unweighted.csv", index=False)
df_sentiment_weighted.to_csv("../data/generated/sentiment_features_weighted.csv", index=False)

print(f"Saved unweighted: {df_sentiment_unweighted.shape}")
print(f"Saved weighted:   {df_sentiment_weighted.shape}")

Saved unweighted: (14421, 13)
Saved weighted:   (14421, 13)
